# Week 102, Day 3: Review: advanced Python

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w102d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def countdown(n):
    while n:
        yield n
        n -= 1

it = countdown(3)
print(next(it), list(it), list(it), sum(countdown(4)))

In [ ]:
from itertools import islice, chain, groupby

def walks():
    yield from [("Jan", 6.0), ("Jan", 3.5), ("Feb", 11.0)]
    yield ("Mar", 15.5)

print(list(islice(chain(walks(), walks()), 5))[-1])
print([(month, sum(km for _, km in group)) for month, group in groupby(walks(), key=lambda w: w[0])])

In [ ]:
import functools

def logged(prefix):
    def decorate(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            print(f"{prefix} {fn.__name__}{args} = {result}")
            return result
        return wrapper
    return decorate

@logged("call:")
def climb(a, b):
    """Metres up from a to b."""
    return max(0, b - a)

climb(40, 310)
print(climb.__name__, climb.__doc__, functools.partial(climb, 40)(80))

In [ ]:
from contextlib import contextmanager

@contextmanager
def opened_section(log, name):
    log.append(f"start {name}")
    try:
        yield log
    finally:
        log.append(f"end {name}")

log = []
try:
    with opened_section(log, "import"):
        raise KeyError("Island")
except KeyError:
    pass
print(log)

In [ ]:
def task(name, steps):
    for i in range(steps):
        yield f"{name} step {i + 1}"

queue = [task("plan", 2), task("share", 3)]
while queue:
    current = queue.pop(0)
    try:
        print(next(current))
        queue.append(current)
    except StopIteration:
        pass

> **Key idea.** Iterables make iterators; generators are lazy, single-use iterators; chain them with `itertools` and `yield from`. Functions are objects: closures (with `nonlocal`), decorators (with `wraps`), decorators with arguments, `partial`, `lru_cache`. Context managers clean up in `__exit__` or `finally`. `__getattr__` handles missing attributes; `__slots__` fixes them. Measure before optimising. Threads for waiting, processes for computing, asyncio for cooperative tasks that yield at each wait.

### Exercise: Lazy chunks

Write a generator `chunked(iterable, n)` yielding lists of up to `n` items from any iterable (including an endless one), using `itertools.islice`, and never reading more than it needs.

In [ ]:
from itertools import islice

def chunked(iterable, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import inspect, itertools\n\ndef test_chunks():\n    \"lazy\"\n    assert list(chunked(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]] and list(chunked([], 2)) == []\n    gen = chunked(itertools.count(), 4)\n    assert inspect.isgenerator(gen) and next(gen) == [0, 1, 2, 3] and next(gen) == [4, 5, 6, 7]")

### Exercise: A decorator with an argument

Write `timed(log, clock)`: a decorator factory whose decorator records, for each call, `(name, seconds)` in the list `log` (seconds from `clock()` before and after, rounded to 3 decimal places), returns the function's result, re-raises its exceptions (still recording the time), and keeps the function's name and docstring.

In [ ]:
import functools

def timed(log, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timed():\n    \"wraps, records, returns\"\n    ticks = iter([0.0, 0.25, 1.0, 1.5])\n    log = []\n    @timed(log, lambda: next(ticks))\n    def plan(a, b):\n        \"\"\"Plan a route.\"\"\"\n        if a == b:\n            raise ValueError(\"same place\")\n        return [a, b]\n    assert plan(\"Mill\", \"Ridge\") == [\"Mill\", \"Ridge\"]\n    try:\n        plan(\"Mill\", \"Mill\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"re-raise\")\n    assert log == [(\"plan\", 0.25), (\"plan\", 0.5)] and plan.__name__ == \"plan\" and plan.__doc__ == \"Plan a route.\"")

### Exercise: A closure that counts

Write `make_tally()`: return a function `add(km)` that adds to a running total kept in the closure (with `nonlocal`) and returns the new total; the returned function also has an attribute `reset` (a function setting the total back to 0).

In [ ]:
def make_tally():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tally():\n    \"nonlocal\"\n    t = make_tally()\n    assert t(6.0) == 6.0 and t(3.5) == 9.5\n    t.reset()\n    assert t(1.0) == 1.0\n    other = make_tally()\n    assert other(2.0) == 2.0 and t(0) == 1.0, \"separate totals\"")

### Exercise: Collecting errors instead of stopping

Write a context manager `collect(errors, kind=ValueError)` with `contextlib.contextmanager`: inside its block, an exception of type `kind` is caught and its message appended to `errors` (the block stops there); other exceptions pass through.

In [ ]:
from contextlib import contextmanager

@contextmanager
def collect(errors, kind=ValueError):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_collect():\n    \"suppress one kind\"\n    errors = []\n    for text in [\"6.0\", \"six\", \"3.5\", \"-\"]:\n        with collect(errors):\n            float(text)\n    assert errors == [\"could not convert string to float: 'six'\", \"could not convert string to float: '-'\"]\n    try:\n        with collect(errors):\n            {}[\"x\"]\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"other errors pass\")")

### Exercise: Settings with defaults

Write a class `Settings(defaults, **given)` using `__slots__ = ("_values",)`: reading `s.name` returns the given value, else the default, through `__getattr__`; an unknown name raises `AttributeError`; setting any attribute raises `AttributeError("settings are read-only")`; and `s.changed()` returns the names given that differ from their defaults, sorted.

In [ ]:
class Settings:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"__getattr__ and __slots__\"\n    defaults = {\"steep_limit\": 80, \"pace\": 12, \"units\": \"km\"}\n    s = Settings(defaults, pace=15, units=\"km\")\n    assert s.steep_limit == 80 and s.pace == 15 and s.changed() == [\"pace\"]\n    for action in (lambda: s.colour, lambda: setattr(s, \"pace\", 20)):\n        try:\n            action()\n        except AttributeError:\n            pass\n        else:\n            raise AssertionError(\"should raise\")\n    assert not hasattr(s, \"__dict__\")")

### Exercise: Taking turns

Write `round_robin(tasks)`: `tasks` maps names to generators; run them in turns (in the order given), taking one value from each in turn until all are exhausted, and return the list of `(name, value)`.

In [ ]:
from collections import deque

def round_robin(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_turns():\n    \"cooperative multitasking\"\n    def steps(n):\n        for i in range(1, n + 1):\n            yield f\"step {i}\"\n    assert round_robin({\"plan\": steps(2), \"share\": steps(3), \"stats\": steps(1)}) == [\n        (\"plan\", \"step 1\"), (\"share\", \"step 1\"), (\"stats\", \"step 1\"), (\"plan\", \"step 2\"), (\"share\", \"step 2\"), (\"share\", \"step 3\")]\n    assert round_robin({}) == []")

### Bug hunt: A decorator that lost the result

The wrapper calls the function but doesn't return what it returned, so every decorated function returns `None` and this prints `None`. Return the result so it prints `270`.

In [ ]:
import functools

def logged(fn):
    @functools.wraps(fn)
    def wrapper(*args):
        fn(*args)
    return wrapper

@logged
def climb(a, b):
    return max(0, b - a)

print(climb(40, 310))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the result\"\n    expect_output(\"270\")")

### Bug hunt: Clean-up that only ran on success

Without `try/finally`, an exception inside the block skips the code after `yield`, so the lock is never released and this prints `['acquire']`. Add `try/finally` so it prints `['acquire', 'release']`.

In [ ]:
from contextlib import contextmanager

events = []

@contextmanager
def lock():
    events.append("acquire")
    yield
    events.append("release")

try:
    with lock():
        raise RuntimeError("boom")
except RuntimeError:
    pass
print(events)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"finally\"\n    expect_output(\"['acquire', 'release']\")")

### Puzzle: A decorator with an argument

Arrange a decorator factory that repeats a function's call `times` times and returns the last result. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # list(chunked("abcde", 2)) as lists of strings
q = ___   # the keyword that lets a closure rebind an outer variable
r = ___   # True or False: __getattr__ is called for every attribute access
s = ___   # what the concurrency model "one thread, tasks that pause at each wait" is called (the module name)
t = ___   # sum(x * x for x in range(4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [[\"a\", \"b\"], [\"c\", \"d\"], [\"e\"]] and q == \"nonlocal\" and r is False, \"p, q and r\"\n    assert s == \"asyncio\" and t == 14, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny event loop

Write `event_loop(tasks)`: `tasks` maps names to generators that **yield a number of seconds to wait** (like `await asyncio.sleep(...)`), and may `return` a message at the end. Simulate time with a heap of `(wake_time, order, name)` (order breaks ties by the order tasks were given and resumed): start every task at time 0; each time a task is resumed it runs to its next `yield`. Return the log: `"T NAME waits S"` for each wait and `"T NAME done: MESSAGE"` (or `"T NAME done"` without a message) when it finishes, with T in the `g` format.

In [ ]:
import heapq

def event_loop(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loop():\n    \"cooperative, by wake time\"\n    def walker(name, legs):\n        for leg in legs:\n            yield leg\n        return f\"{name} arrived\"\n    def reminder():\n        yield 1.5\n        return None\n    log = event_loop({\"ada\": walker(\"Ada\", [2, 1]), \"bo\": walker(\"Bo\", [1, 3]), \"ping\": reminder()})\n    assert log == [\"0 ada waits 2\", \"0 bo waits 1\", \"0 ping waits 1.5\", \"1 bo waits 3\", \"1.5 ping done\", \"2 ada waits 1\",\n                   \"3 ada done: Ada arrived\", \"4 bo done: Bo arrived\"]\n    assert event_loop({}) == []")

## Warm-up

A generator runs only when asked for the next value, remembers where it stopped, and is used up after one pass. Most of Phase 5 is variations on that theme: objects that do their work at the moment it's needed.

## Iteration

`for x in obj` calls `iter(obj)` once, then `next()` until `StopIteration`. An **iterable** makes iterators (`__iter__`); an **iterator** also has `__next__` and returns itself from `__iter__`. Generator functions (with `yield`) and generator expressions are the easy way to write iterators. `itertools` (`islice`, `chain`, `groupby`, `accumulate`) and `yield from` build lazy pipelines that never hold everything in memory.

## Functions as objects

Functions are values: pass them, return them, store them. A **closure** remembers variables from the function that made it (`nonlocal` to rebind them). A **decorator** is a function that takes a function and returns a replacement; decorators with arguments are functions returning decorators. Use `functools.wraps` to keep the name and docstring; `functools.partial` fixes some arguments; `lru_cache` memoises.

## Context managers and attributes

`with` guarantees clean-up: `__exit__` (or the code after `yield` in a `contextlib.contextmanager` generator, inside `try/finally`) runs however the block ends. Attribute access: `__getattr__` is called only when normal lookup fails (good for defaults and proxies); properties and descriptors control individual attributes; `__slots__` fixes the attributes an instance may have and saves memory.

## Memory, performance and concurrency

Names refer to objects; assignment never copies; CPython frees objects when their reference count reaches zero and collects cycles with `gc`. Performance: measure first (`timeit`, `cProfile`), then fix the algorithm, then use the right built-ins. Concurrency: **threads** share memory and suit waiting on I/O (the GIL lets one run Python bytecode at a time); **processes** suit computation; **asyncio** runs many tasks cooperatively on one thread: each task runs until it `await`s something, then another runs. A generator that `yield`s at each wait is the same idea in miniature.

## Practice

## Quiz

## Daily challenge

## Before you go

Tomorrow reviews Phase 6's engineering: testing, packaging, quality, databases, the web, automation, security, system design and data.